In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# =========================
# 1. Imports
# =========================
import os, json, re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader

# =========================
# 2. LOAD JSONL DATA
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/gru-primevul26"

def load_jsonl(file):
    data = []
    with open(file, "r") as f:
        for line in f:
            obj = json.loads(line)

            code = (
                obj.get("func_before") or
                obj.get("code") or
                obj.get("func") or
                ""
            )

            label = obj.get("target", obj.get("label", 0))

            if code.strip():
                data.append((code, int(label)))

    return data

train_data = load_jsonl(os.path.join(base_path, "primevul_train_paired.jsonl"))
val_data   = load_jsonl(os.path.join(base_path, "primevul_valid_paired.jsonl"))
test_data  = load_jsonl(os.path.join(base_path, "primevul_test_paired.jsonl"))

# Merge train + validation
train_data += val_data

X_train = [x[0] for x in train_data]
y_train = [x[1] for x in train_data]

X_test  = [x[0] for x in test_data]
y_test  = [x[1] for x in test_data]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# =========================
# 3. TOKENIZATION
# =========================
def tokenize(text):
    text = text.lower()
    text = re.sub(r'([(){}\[\];,<>!=&|^~*/%+-])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.split()

counter = Counter()
for t in X_train:
    counter.update(tokenize(t))

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, freq in counter.most_common(20000):
    if freq >= 2:
        vocab[word] = len(vocab)

def encode(text):
    return [vocab.get(w, 1) for w in tokenize(text)]

MAX_LEN = 300

def pad(seq):
    return seq[:MAX_LEN] + [0]*(MAX_LEN - len(seq))

X_train = [pad(encode(t)) for t in X_train]
X_test  = [pad(encode(t)) for t in X_test]

# =========================
# 4. DATASET CLASS
# =========================
class PrimeVulDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(PrimeVulDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(PrimeVulDataset(X_test, y_test), batch_size=32)

# =========================
# 5. GRU MODEL
# =========================
class GRUModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)

        self.gru = nn.GRU(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(512, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.gru(x)

        out = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(out)).squeeze()

# =========================
# 6. SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUModel(len(vocab)).to(device)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

pos_weight = torch.tensor(class_weights[1]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# =========================
# 7. TRAINING
# =========================
for epoch in range(8):
    model.train()
    total_loss = 0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# =========================
# 8. EVALUATION
# =========================
model.eval()
preds, true = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        out = torch.sigmoid(model(Xb.to(device)))
        preds.extend((out > 0.5).cpu().numpy())
        true.extend(yb.numpy())

acc = accuracy_score(true, preds)
report = classification_report(true, preds, output_dict=True)

print("\nAccuracy:", acc)
print("\nClassification Report:\n", classification_report(true, preds))

# =========================
# 9. SAVE RESULTS
# =========================
label_key = [k for k in report.keys() if k.startswith("1")][0]

results = {
    "Model": "GRU",
    "Dataset": "PrimeVul",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score']
}

pd.DataFrame([results]).to_csv("/kaggle/working/gru_primevul_results.csv", index=False)

print("\n✅ Results saved")

Train size: 8538
Test size: 870
Epoch 1, Loss: 0.6975
Epoch 2, Loss: 0.6955
Epoch 3, Loss: 0.6952
Epoch 4, Loss: 0.6940
Epoch 5, Loss: 0.6933
Epoch 6, Loss: 0.6928
Epoch 7, Loss: 0.6927
Epoch 8, Loss: 0.6913

Accuracy: 0.5160919540229885

Classification Report:
               precision    recall  f1-score   support

         0.0       0.52      0.43      0.47       435
         1.0       0.51      0.60      0.56       435

    accuracy                           0.52       870
   macro avg       0.52      0.52      0.51       870
weighted avg       0.52      0.52      0.51       870


✅ Results saved
